In [1]:
import pandas as pd
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)

df = pd.read_csv("../data/processed/nft_confidence_filtered.csv")

df["label_final"] = df["label_final"].astype(int)
df = df.sort_values("timestamp").reset_index(drop=True)

print(df.shape)
print(df["label_final"].value_counts())

C:\Users\VENTUS\AppData\Local\Temp\ipykernel_21632\2361970566.py:6: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/nft_confidence_filtered.csv")


(1490796, 25)
label_final
0    1489978
1        818
Name: count, dtype: int64


# Splitting

In [2]:
split_idx = int(len(df) * 0.8)

train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

print("Train:", train_df.shape)
print(train_df["label_final"].value_counts())

print("Test:", test_df.shape)
print(test_df["label_final"].value_counts())

Train: (1192636, 25)
label_final
0    1191882
1        754
Name: count, dtype: int64
Test: (298160, 25)
label_final
0    298096
1        64
Name: count, dtype: int64


# Undersampling data train

In [3]:
MAX_RATIO = 50

wash_train = train_df[train_df["label_final"] == 1].copy()
normal_train = train_df[train_df["label_final"] == 0].copy()

target_normal = len(wash_train) * MAX_RATIO

print("Wash train:", len(wash_train))
print("Normal train before undersampling:", len(normal_train))
print("Target normal:", target_normal)

Wash train: 754
Normal train before undersampling: 1191882
Target normal: 37700


In [4]:
normal_train_sampled = normal_train.sample(
    n=target_normal,
    random_state=42
)

In [5]:
train_1_50 = pd.concat(
    [wash_train, normal_train_sampled],
    axis=0
)

train_1_50 = train_1_50.sort_values("timestamp").reset_index(drop=True)

print(train_1_50["label_final"].value_counts())

label_final
0    37700
1      754
Name: count, dtype: int64


In [6]:
wash_count = (train_1_50["label_final"] == 1).sum()
normal_count = (train_1_50["label_final"] == 0).sum()

print("Final train wash:", wash_count)
print("Final train normal:", normal_count)
print("Final train ratio normal/wash:", normal_count / wash_count)

Final train wash: 754
Final train normal: 37700
Final train ratio normal/wash: 50.0


In [7]:
test_df = test_df.sort_values("timestamp").reset_index(drop=True)

print("Test distribution:")
print(test_df["label_final"].value_counts())

Test distribution:
label_final
0    298096
1        64
Name: count, dtype: int64


In [8]:
train_1_50.to_csv(
    "../data/processed/train_1_50.csv",
    index=False
)

test_df.to_csv(
    "../data/processed/test_temporal.csv",
    index=False
)

train_df.to_csv(
    "../data/processed/train_temporal_original.csv",
    index=False
)

print("Saved:")
print("../data/processed/train_1_50.csv")
print("../data/processed/test_temporal.csv")
print("../data/processed/train_temporal_original.csv")

Saved:
../data/processed/train_1_50.csv
../data/processed/test_temporal.csv
../data/processed/train_temporal_original.csv


In [ ]:
Karena data transaksi blockchain bersifat temporal dan sequential, dilakukan temporal split terlebih dahulu sebelum proses undersampling untuk menghindari temporal leakage.

Dataset diurutkan berdasarkan timestamp, kemudian dibagi menggunakan rasio:

Split	        Persentase
Training Set	80%
Testing Set	    20%

Distribusi hasil temporal split:

Dataset	Normal	    Wash Trading
Train	1.191.882	754
Test	298.096	    64

Untuk menangani class imbalance, dilakukan random undersampling hanya pada training set dengan target rasio maksimum 1:50 sesuai hasil konsultasi penelitian.

Jumlah akhir training set setelah undersampling:

Class	        Jumlah
Normal	        37.700
Wash Trading	754

Rasio akhir training set:

1 : 50

Testing set tidak dilakukan undersampling agar evaluasi tetap merepresentasikan distribusi transaksi sebenarnya pada kondisi real-world.

Pendekatan ini dipilih untuk:

1. Menghindari temporal leakage.
2. Menjaga realism distribusi testing.
3. Mengontrol class imbalance pada training graph model.
4. Meningkatkan stabilitas proses training.